# Running this tutorial in GitHub Codespaces
Wait for setup to finish, then select **CPS Tutorial (Python 3.12)** as the notebook kernel. Run cells in order from the repository root. Dependencies are already installed.

The anomaly-detection example uses the supplied **35% pruned model**. It retains the original 30-example encrypted inference, which can take substantial time. For a quicker first pass, set `run_fhe = 0`; compilation still occurs. Exercise placeholders are intentionally left for students. See `README.md`.


# CPS Tutorial - Part 2 - TFHE scheme through Zama Concrete

by Apostolos Fournaris (Industrial System Institute/R.C. ATHENA), Paolo Palmieri (UCC) and Francesco Regazzoni (UvA)

SECURED project (Horizon Europe grant no. 101095717) 

What is Concrete

Concrete is an open-source FHE Compiler that simplifies the use of fully homomorphic encryption (FHE), developed by Zama.

As you have seen in Part 1 of this tutorial, Homomorphic Encryption (HE) enables performing computations on encrypted data directly without the need to decrypt it first. FHE allows developers to build services that ensure privacy for all users. FHE is also an excellent solution against data breaches as everything is performed on encrypted data. Even if the server is compromised, no sensitive data will be leaked.

Concrete is a versatile library that can be used for a variety of purposes. For instance, Concrete ML is built on top of Concrete to simplify Machine-Learning oriented use cases.

Let's start by installing the library:

In [ ]:
# Dependencies are installed by the Codespaces setup.
from importlib.metadata import version
print("Concrete:", version("concrete-python"))
print("Concrete ML:", version("concrete-ml"))


## Concrete Basics

In Concrete, there are essentially two types of operations:

**Linear operations**, such as addition, subtraction, and multiplication by an integer, which are very fast.

**All other operations**, which are performed through table lookups (TLU).

In Concrete, the supported **non-linear operations** primarily revolve around table lookups (TLUs). Non-linear operations are those that cannot be represented as simple additions, subtractions, or multiplications by integers, which are considered linear operations.

Here are the key **non-linear operations** supported by Concrete FHE through table lookups:

1. **Comparison operations**:
   - Equality (`==`)
   - Greater than (`>`)
   - Less than (`<`)
   - Greater than or equal (`>=`)
   - Less than or equal (`<=`)

2. **Logical operations**:
   - Logical AND (`&&`)
   - Logical OR (`||`)
   - Logical NOT (`!`)

3. **Polynomial functions**:
   - Concrete FHE can approximate certain non-linear polynomial functions using TLUs, like squaring (`x^2`), cubic functions, etc.

4. **Activation functions** (useful for encrypted machine learning):
   - Functions like ReLU (Rectified Linear Unit), sigmoid, and others can be supported through lookup tables.

5. **Modulo operations**:
   - Non-linear modular arithmetic operations like `mod` can be supported by TLUs.

While Concrete supports fast linear operations natively, any operation beyond this, such as complex mathematical functions, logic, or comparison, is typically mapped to TLUs, which execute the non-linear operations by precomputing and storing results in tables. These tables can vary in size depending on the bitwidth of the input, which affects performance.

More on Concrete non-linear operations:https://docs.zama.ai/concrete/core-features/non_linear_operations

In this first example, we compute a simple function (addition) in a homomorphic manner:

In [ ]:
from concrete import fhe

def add(x, y): 
    return x + (y**2)

compiler = fhe.Compiler(add, {"x": "encrypted", "y": "encrypted"})

print(add(13,4))

import numpy as np
rng = np.random.default_rng(42)
# Include both range boundaries explicitly for reliable compilation.
inputset = [(0, 0), (255, 15)] + [
    (int(rng.integers(0, 256)), int(rng.integers(0, 16))) for _ in range(48)
]
assert len(inputset) == 50


print(f"Compilation...")
circuit = compiler.compile(inputset)

print(f"Key generation...")
circuit.keygen()

print(f"Homomorphic evaluation...")
encrypted_x, encrypted_y = circuit.encrypt(13, 4)
encrypted_result = circuit.run(encrypted_x, encrypted_y)
result = circuit.decrypt(encrypted_result)
print(encrypted_x)
assert result == add(13, 4)
print(f"Result: {result}")


<hr>

## Modifying the example:
Let's try to modify the above example by  adding **non-linear operations** to it.
Instead of always performing $x + (y^2)$, we want to perform $x + y^2$ if the value of $y$ is less than 3, or $x + y$ if the value od $y$ is greater or equal to  3.

In [ ]:
#your code to be added here
from concrete import fhe

<hr>

<hr>

## Concrete ML overview

**Concrete-ML** is an open-source framework developed by **Zama** that enables privacy-preserving machine learning (ML) through the use of **Fully Homomorphic Encryption (FHE)**. It allows ML models to be trained and executed on encrypted data, ensuring that sensitive information remains private even during inference. The core idea is to run machine learning algorithms on encrypted data without decrypting it, thus providing strong privacy guarantees, especially for sensitive data.

### Key Features and Components of Concrete-ML:

1. **Privacy-Preserving Inference**:
   - The main objective of Concrete-ML is to perform **inference on encrypted data**. Once a model is trained, Concrete-ML can take encrypted inputs and perform predictions without decrypting the data, ensuring that private information stays secure throughout the process.

2. **Integration with Popular ML Libraries**:
   - Concrete-ML integrates with widely-used machine learning libraries such as **Scikit-learn** and **PyTorch**. This makes it user-friendly for data scientists and machine learning engineers who can easily convert models into FHE-compatible versions.

3. **Homomorphic Encryption (FHE)**:
   - The core technology behind Concrete-ML is **Fully Homomorphic Encryption** (FHE), a cryptographic method that allows computations on encrypted data. Concrete-ML is built on Zama’s **Concrete** FHE library, which optimizes the encryption and decryption processes to enable practical use cases.
   
4. **Automated Conversion to FHE**:
   - Concrete-ML provides an easy pipeline to convert traditional ML models into **FHE-compatible models**. It handles the complexities of translating non-linear operations into **table lookups (TLUs)**, which are needed to perform non-linear computations in an encrypted domain.

5. **Support for Classical ML Models**:
   - Concrete-ML focuses on privacy-preserving versions of traditional machine learning algorithms. Some supported algorithms include:
     - **Linear models**: Logistic regression, linear regression
     - **Tree-based models**: Decision trees, Random forests
     - **Neural Networks**: Simple feedforward networks with FHE-compatible activation functions
   - The framework is designed to optimize and adapt models for FHE, allowing them to run efficiently on encrypted data.

6. **Inference on Encrypted Data**:
   - Once the model is trained, Concrete-ML transforms the model into a format that can perform predictions on encrypted data. The output of the model remains encrypted and can only be decrypted by the owner of the private key.

7. **Efficiency and Performance Optimizations**:
   - Running machine learning models on encrypted data can be computationally intensive. Concrete-ML optimizes the performance of FHE operations, including minimizing the complexity of table lookups (used for non-linear functions) and efficiently handling linear operations (which are faster in FHE).

8. **Security and Privacy**:
   - Concrete-ML leverages the strong security guarantees of FHE to ensure that data remains private throughout the computation process. Data is never decrypted during inference, meaning the model, input, and predictions remain confidential.


Let's now install the Concrete-ML library, to run a more complex example based on a simple Machine Learning case:

In [ ]:
# Dependencies are installed by the Codespaces setup.
from importlib.metadata import version
print("Concrete:", version("concrete-python"))
print("Concrete ML:", version("concrete-ml"))


The example below does something related to ML, in an encrypted form:

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from concrete.ml.sklearn import LogisticRegression

# Lets create a synthetic data-set
x, y = make_classification(n_samples=100, class_sep=2, n_features=30, random_state=42)

# Split the data-set into a train and test set
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

# Now we train in the clear and quantize the weights
model = LogisticRegression(n_bits=8)
model.fit(X_train, y_train)

# We can simulate the predictions in the clear
y_pred_clear = model.predict(X_test)

# We then compile on a representative set
model.compile(X_train)

# Finally we run the inference on encrypted inputs !
y_pred_fhe = model.predict(X_test, fhe="execute")

print("In clear  :", y_pred_clear)
print("In FHE    :", y_pred_fhe)
print(f"Similarity: {int((y_pred_fhe == y_pred_clear).mean()*100)}%")

# Output:
    # In clear  : [0 0 0 0 1 0 1 0 1 1 0 0 1 0 0 1 1 1 0 0]
    # In FHE    : [0 0 0 0 1 0 1 0 1 1 0 0 1 0 0 1 1 1 0 0]
    # Similarity: 100%

In [ ]:
import onnx
import numpy as np
import time

from concrete.ml.torch.compile import compile_onnx_model

# Parameters
start_idx = 2 #Number of 32-inputs to bypass
items = 30 # ~6000
# max 7 bits of quantization
quant_bits = 3
run_fhe = 1 #0: just evaluate 1: run fhe model
# Load input
start = start_idx * 32+1
input_set = np.load('./test_X.npy')
output_set = np.load('./test_Y.npy')
#indices = range(start,start+items)

# Set the maximum number based on the dataset size
max_number = output_set.shape[0] - 1  # Adjust to the maximum index of output_set
np.random.seed(42)  # Replace 42 with any integer
indices = np.random.choice(range(max_number + 1), size=30, replace=False)
print("Generated indices:", indices)
print("Target values are:", output_set[indices])
for i in indices:
        print(f"Indices[{i}]")

print("Target values are:", output_set[indices])



print("Input data set size:",input_set.size)
print("Output data set size:",output_set.size)
#print("Reference output: ", output_set[indices])
print("-----------------------------------")
# Load model from onnx type
#onnx_model = onnx.load("../Anomaly_detection/model_ann1_v14.onnx") 
# pruned version
onnx_model = onnx.load("./pruned_model_35_percent_acc_9629.onnx")
onnx.checker.check_model(onnx_model)
print("Loaded datasets and onnx model")
print("-------------------------------")
print("Quantization bits: ",quant_bits)
print("-------------------------------")
# Compile
print("Compiling model...")
start_time = time.time()
quantized_module = compile_onnx_model(
    onnx_model, input_set, n_bits=quant_bits
)
print("Compiled model in --- %s seconds ---" % (time.time() - start_time))

#Run model
#print("Reference output: ", output_set[indices])
print("-----------------------------------")
print("Running clean model")
start_time = time.time()
y_clear = quantized_module.forward(input_set[indices], fhe="disable")
print("Execution in clear --- %s seconds ---" % ((time.time() - start_time)/items))
quant_y_clear = (y_clear>0.5).astype(int)
#       print("Outputs size: ", y_clear.size)
#       print("Quantized outputs size:",quant_y_clear.size)
print("Outputs in clear: ", y_clear)
print("Outputs in q_clear: ", quant_y_clear)
sum = 0
for i in range(items):
        if(quant_y_clear[i] == output_set[indices[i]]):
                sum = sum + 1
print("Accuracy:", (sum/items)*100, "%")
if(run_fhe):
        print("Running FHE model")
        start_time = time.time()
        y_fhe = quantized_module.forward(input_set[indices], fhe="execute")
        print("Execution in FHE --- %s seconds ---" % ((time.time() - start_time)/items))
        print("Outputs in FHE:   ", y_fhe)                    


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=e242c3b6-635b-4ae1-be8d-6a06c9f5d533' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>